In [12]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from pathlib import Path

In [13]:
project_path = Path("/Users/irinafendley/Projects/Loan_Default")

df = pd.read_csv(
    project_path / "data/processed/loan_clean.csv"
)

In [14]:
model = joblib.load(
    project_path / "data/models/logistic_regression.pkl"
)

In [15]:
# risky segment definition
risk_group = df[
    (df["CreditScore"] < 400) &
    (df["Income"] < df["Income"].quantile(0.25)) &
    (df["DTIRatio"] > 0.55)
].copy()

In [16]:
risk_group_summary = {
    "Count": len(risk_group),
    "Default_rate_%": risk_group["Default"].mean() * 100,
    "Total_loan": risk_group["LoanAmount"].sum(),
    "Total_loss": (risk_group["LoanAmount"] * risk_group["Default"]).sum()
}

print(risk_group_summary)

{'Count': 5071, 'Default_rate_%': np.float64(20.19325576809308), 'Total_loan': np.int64(645561822), 'Total_loss': np.int64(159711094)}


In [17]:
baseline_default_rate = df["Default"].mean() * 100
baseline_volume = df["LoanAmount"].sum()

print("Baseline default rate:", baseline_default_rate)
print("Baseline volume:", baseline_volume)

Baseline default rate: 11.612824901017047
Baseline volume: 32576880572


In [22]:
rejected_mask = risk_group.index

sim_portfolio = df.drop(index=rejected_mask).copy()

In [19]:
new_default_rate = sim_portfolio["Default"].mean() * 100
new_volume = sim_portfolio["LoanAmount"].sum()

print("After rejection policy:")
print("Default rate:", new_default_rate)
print("Volume:", new_volume)
print("Reduction in volume:", baseline_volume - new_volume)
print("Reduction in default rate:", baseline_default_rate - new_default_rate)

After rejection policy:
Default rate: 11.438971375601337
Volume: 31931318750
Reduction in volume: 645561822
Reduction in default rate: 0.1738535254157103


In [20]:
lost_clients = len(risk_group)
lost_volume = risk_group["LoanAmount"].sum()

print("Rejected clients:", lost_clients)
print("Lost volume:", lost_volume)

Rejected clients: 5071
Lost volume: 645561822


In [27]:
loss_before = (df["LoanAmount"] * df["Default"]).sum()

df_after = df.copy()
df_after.loc[rejected_mask, "LoanAmount"] = 0

loss_after = (df_after["LoanAmount"] * df_after["Default"]).sum()

savings = loss_before - loss_after

savings_pct = savings / loss_before * 100

print(f"Expected default loss for the entire loan portfolio (before policy): ${loss_before:,.0f}")
print(f"Expected default loss for the entire loan portfolio (after rejecting the high-risk segment): ${loss_after:,.0f}")
print(f"Expected loss avoided by the rejection policy: ${savings:,.0f}")
print(f"Reduction in expected portfolio loss: {savings_pct:.2f}%")

Expected default loss for the entire loan portfolio (before policy): $4,285,312,531
Expected default loss for the entire loan portfolio (after rejecting the high-risk segment): $4,125,601,437
Expected loss avoided by the rejection policy: $159,711,094
Reduction in expected portfolio loss: 3.73%


In [26]:
default_rate_before = df["Default"].mean() * 100

sim_portfolio = df.drop(index=rejected_mask)

default_rate_after = sim_portfolio["Default"].mean() * 100

print(f"Portfolio default rate before: {default_rate_before:.2f}%")
print(f"Portfolio default rate after: {default_rate_after:.2f}%")
print(f"Reduction: {default_rate_before - default_rate_after:.2f} percentage points")

Portfolio default rate before: 11.61%
Portfolio default rate after: 11.44%
Reduction: 0.17 percentage points
